# 11 - Complaint Sentiment & NLP Analysis
LLM-powered summarization, classification, and emotion detection using Groq.

In [ ]:
import pandas as pd, numpy as np, os, time
from dotenv import load_dotenv
load_dotenv()
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'
import plotly.express as px


In [ ]:
# LLM setup
provider = os.getenv("LLM_PROVIDER", "groq")
print(f"LLM Provider: {provider}")

if provider == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1,
                   api_key=os.getenv("GROQ_API_KEY"))
elif provider == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)
else:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Quick test
resp = llm.invoke("Say 'LLM connection OK' and nothing else.")
print(f"LLM test: {resp.content.strip()}")


In [ ]:
# Load data and sample
df = pd.read_csv('data/processed/complaints_clean.csv')
print(f"Full shape: {df.shape}")

sample_df = df.dropna(subset=['narrative'])
sample_df = sample_df[sample_df['narrative'].str.len() > 50]
sample_df = sample_df.sample(n=min(500, len(sample_df)), random_state=SEED).reset_index(drop=True)
print(f"Sample shape: {sample_df.shape}")


In [ ]:
# STEP 1: Complaint Summarization
summaries = []
checkpoint_path = 'data/processed/complaints_nlp_checkpoint.csv'

for i, row in sample_df.iterrows():
    prompt = f"Summarize this customer financial complaint in 1-2 sentences, focusing on the core issue: {str(row['narrative'])[:1000]}"
    try:
        response = llm.invoke(prompt)
        result = response.content.strip()
    except Exception as e:
        print(f"Error at row {i}: {e}")
        time.sleep(2)
        try:
            response = llm.invoke(prompt[:500])
            result = response.content.strip()
        except:
            result = "Summary unavailable"
    summaries.append(result)
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        print(f"Summarization checkpoint: {i+1}/{len(sample_df)}")

sample_df['complaint_summary'] = summaries
print(f"Summarization complete. Sample: {summaries[0][:100]}...")


In [ ]:
# STEP 2: Classification
categories_list = []
for i, row in sample_df.iterrows():
    prompt = (f"Classify this financial complaint into exactly ONE category. "
              f"Categories: Billing, Fraud, Card Declined, Rewards, Customer Service, Service Delay, Credit Reporting, Collections. "
              f"Return ONLY the category name with no other text. "
              f"Complaint: {str(row['narrative'])[:800]}")
    try:
        response = llm.invoke(prompt)
        result = response.content.strip()
    except:
        time.sleep(2)
        try:
            response = llm.invoke(prompt[:400])
            result = response.content.strip()
        except:
            result = "Unknown"
    categories_list.append(result)
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        print(f"Classification checkpoint: {i+1}/{len(sample_df)}")

sample_df['complaint_category'] = categories_list
print(f"Classification complete.")
print(sample_df['complaint_category'].value_counts())


In [ ]:
# STEP 3: Emotion Detection
emotions = []
for i, row in sample_df.iterrows():
    prompt = (f"Detect the dominant emotion in this financial complaint. "
              f"Return ONLY one word from: Anger, Frustration, Neutral, Legal Threat, Distress. "
              f"No other text. Complaint: {str(row['narrative'])[:800]}")
    try:
        response = llm.invoke(prompt)
        result = response.content.strip()
    except:
        time.sleep(2)
        try:
            response = llm.invoke(prompt[:400])
            result = response.content.strip()
        except:
            result = "Neutral"
    emotions.append(result)
    time.sleep(0.3)
    if (i + 1) % 50 == 0:
        print(f"Emotion checkpoint: {i+1}/{len(sample_df)}")

sample_df['emotion'] = emotions
print(f"Emotion detection complete.")
print(sample_df['emotion'].value_counts())


In [ ]:
fig = px.pie(sample_df, names='complaint_category', title='Complaint Category Distribution',
             hole=0.3, template=TEMPLATE)
fig.show()

cat_counts = sample_df['complaint_category'].value_counts().reset_index()
cat_counts.columns = ['category','count']
fig = px.bar(cat_counts, x='category', y='count', color_discrete_sequence=[PRIMARY],
             template=TEMPLATE, title='Complaint Categories')
fig.update_xaxes(tickangle=45)
fig.show()


In [ ]:
emo = sample_df['emotion'].value_counts().reset_index(); emo.columns=['emotion','count']
fig = px.bar(emo, x='emotion', y='count', color_discrete_sequence=[RISK],
             template=TEMPLATE, title='Emotion Distribution')
fig.show()


In [ ]:
try:
    pivot = sample_df.groupby(['complaint_category','emotion']).size().unstack(fill_value=0)
    fig = px.imshow(pivot, color_continuous_scale='Purples', template=TEMPLATE,
                    title='Category x Emotion Heatmap', text_auto=True)
    fig.show()
except Exception as e:
    print(f"Heatmap error: {e}")


In [ ]:
fig = px.histogram(sample_df, x='narrative_length', color='emotion',
                   template=TEMPLATE, title='Narrative Length by Emotion', nbins=30)
fig.show()


In [ ]:
if 'timely_response' in sample_df.columns:
    ct = sample_df.groupby(['complaint_category','timely_response']).size().reset_index(name='count')
    fig = px.bar(ct, x='complaint_category', y='count', color='timely_response', barmode='group',
                 color_discrete_map={'Yes':SAFE,'No':RISK},
                 template=TEMPLATE, title='Timely Response by Category')
    fig.update_xaxes(tickangle=45)
    fig.show()


In [ ]:
# Save
sample_df.to_csv('data/processed/complaints_with_nlp.csv', index=False)
print(f"Saved complaints_with_nlp.csv - shape: {sample_df.shape}")
